In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X_full = np.load('../data/X_train.npy')
y_full = np.load('../data/y_train.npy')
obs_count_full = np.load('../data/obs_count.npy')
obs_sse_full = np.load('../data/obs_sse.npy')

In [ ]:
n = X_full.shape[0]
n_train = (n * 4) // 5

np.random.seed(305)
shuffled_idx = np.random.permutation(range(n))
train_idx = shuffled_idx[:n_train]
test_idx = shuffled_idx[n_train:]

In [ ]:
np.save('X_train.npy', X_train:=X_full[train_idx])
np.save('y_train.npy', y_train:=y_full[train_idx])
np.save('obs_count.npy', obs_count_full[train_idx])
np.save('obs_sse.npy', obs_sse_full[train_idx])

np.save('X_test.npy', X_test:=X_full[test_idx])
np.save('y_test.npy', y_test:=y_full[test_idx])

In [ ]:
from itertools import product

ls_xy_vals  = [0.015, 0.03, 0.06, 0.12, 0.24]
ls_z_vals   = [0.25, 0.5, 1.0]
os_xyz_vals = [25, 75, 150]
ls_t_vals   = [0.02, 0.1, 0.5, 5.0, 50.0]
os_t_vals   = [5, 25, 75]

grid = list(product(ls_xy_vals, ls_z_vals, os_xyz_vals, ls_t_vals, os_t_vals))
param_grid = pd.DataFrame(grid, columns=["ls_xy", "ls_z", "os_xyz", "ls_t", "os_t"])
param_grid.to_csv('param_grid2.csv', index=None)

In [ ]:
param_grid.shape

# KNN Baseline

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
X_geo_tr = np.load('../data/geo_X_train.npy')
y_geo_tr = np.load('../data/geo_y_train.npy')
X_geo_val = np.load('../data/geo_X_val.npy')
y_geo_val = np.load('../data/geo_y_val.npy')

In [ ]:
param_grid = {
    "n_neighbors": np.arange(1, 21),
    "weights": ["uniform", "distance"],
    "p": [1, 2]  # Manhattan vs Euclidean
}

knn_grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",  # or "r2"
    n_jobs=-1,   # use all CPU cores
    verbose=0
)

knn_grid.fit(X_train, y_train)
print(f"n_neighbors: {knn_grid.best_estimator_.n_neighbors.item()}")

In [ ]:
knn_grid.best_score_

In [ ]:
preds = knn_grid.predict(X_test)
sq_errs = (preds - y_test)**2
mse = sq_errs.mean()

plt.hist(sq_errs)
plt.title(f"MSE: {mse:.4f}")
plt.show()